
#### Recruitment Funnel Analytics - cleaning, EDA and business analysis.

In [ ]:
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10, "axes.titleweight": "bold"})
ACCENT = "#2E5E8C"

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/heryttage/recruitment_analytics/refs/heads/main/data/recruitment_applications.csv")
raw_rows = len(df)

In [ ]:
# ---------- DATA CLEANING ----------
df["department"] = df["department"].str.strip()
df["source_channel"] = (df["source_channel"].str.strip().str.title()
                        .replace({"Linkedin": "LinkedIn", "University/Campus": "University/Campus"}))
df["region"] = df["region"].fillna("Unknown")
df["education_level"] = df["education_level"].fillna("Unknown")
clean_rows = len(df)

print(f"Cleaning: {raw_rows} -> {clean_rows} rows ({raw_rows-clean_rows} dupes removed)")
print(f"Bad/missing dates coerced: {df['application_date'].isna().sum()}")

In [ ]:
# fix known title casing
df["source_channel"] = df["source_channel"].replace({"Linkedin": "LinkedIn"})
# parse dates, coerce typos to NaT
df["application_date"] = pd.to_datetime(df["application_date"], errors="coerce")
# negative experience -> NaN then impute by level median
df.loc[df["years_experience"] < 0, "years_experience"] = np.nan
df["years_experience"] = df.groupby("job_level")["years_experience"].transform(
    lambda s: s.fillna(s.median()))

In [ ]:
# ---------- FUNNEL ----------
total = len(df)
screened = int(df["passed_screening"].sum())
assessment = int((df["final_stage"].isin(["Assessment","Interview","Offer","Hired"])).sum())
interview = int((df["final_stage"].isin(["Interview","Offer","Hired"])).sum())
offer = int(df["offer_extended"].sum())
hired = int(df["hired"].sum())

funnel = pd.Series({"Applied": total, "Screened": screened, "Assessment": assessment,
                    "Interview": interview, "Offer": offer, "Hired": hired})
print("\n=== FUNNEL ===")
for k, v in funnel.items():
    print(f"{k:11s}: {v:5d}  ({v/total*100:5.1f}% of applied)")
print(f"Offer acceptance rate: {df['offer_accepted'].sum()/offer*100:.1f}%")
print(f"Overall applied->hired: {hired/total*100:.2f}%")

In [ ]:
# ---------- SOURCE QUALITY ----------
src = df.groupby("source_channel").agg(
    applications=("application_id","count"),
    hires=("hired","sum"),
    avg_cost=("cost_per_applicant","mean")).assign(
    hire_rate=lambda x: x.hires/x.applications*100,
    cost_per_hire=lambda x: x.avg_cost*x.applications/x.hires).sort_values("hire_rate", ascending=False)
print("\n=== SOURCE QUALITY ===")
print(src.round(1).to_string())

In [ ]:
# ---------- DEPARTMENT ----------
dept = df.groupby("department").agg(
    applications=("application_id","count"),
    hires=("hired","sum"),
    avg_tth=("time_to_hire_days","mean")).assign(
    hire_rate=lambda x: x.hires/x.applications*100).sort_values("hires", ascending=False)
print("\n=== DEPARTMENT ===")
print(dept.round(1).to_string())

In [ ]:

# ---------- TIME TO HIRE ----------
tth = df.loc[df["hired"], "time_to_hire_days"]
print(f"\nTime to hire (days): mean {tth.mean():.1f}, median {tth.median():.0f}, p90 {tth.quantile(.9):.0f}")
# cost
total_cost = df["cost_per_applicant"].sum()
print(f"Total recruiting spend: ${total_cost:,.0f} | Cost per hire: ${total_cost/hired:,.0f}")

In [ ]:
# ---------- CHARTS ----------
# Funnel chart
fig, ax = plt.subplots(figsize=(8,4.5))
colors = plt.cm.Blues(np.linspace(0.45,0.9,len(funnel)))
ax.barh(range(len(funnel)), funnel.values, color=colors)
ax.set_yticks(range(len(funnel))); ax.set_yticklabels(funnel.index)
ax.invert_yaxis()
for i,(k,v) in enumerate(funnel.items()):
    ax.text(v+10, i, f"{v}  ({v/total*100:.0f}%)", va="center", fontsize=9)
ax.set_title("Recruitment Funnel — Applied to Hired"); ax.set_xlabel("Candidates")
plt.tight_layout(); plt.savefig("01_funnel.png"); #plt.close()
plt.show()

In [ ]:
# Source: hire rate vs cost per hire
fig, ax = plt.subplots(figsize=(8,4.5))
ax.scatter(src["cost_per_hire"], src["hire_rate"], s=src["applications"]*1.2,
           color=ACCENT, alpha=.7, edgecolor="white")
for name,r in src.iterrows():
    ax.annotate(name, (r["cost_per_hire"], r["hire_rate"]), fontsize=8,
                xytext=(5,4), textcoords="offset points")
ax.set_xlabel("Cost per hire ($)"); ax.set_ylabel("Hire rate (%)")
ax.set_title("Channel efficiency — hire rate vs cost per hire\n(bubble = application volume)")
plt.tight_layout(); plt.savefig("02_source_efficiency.png"); #plt.close()
plt.show()

In [ ]:
# Dept hire rate
fig, ax = plt.subplots(figsize=(8,4.5))
dd = dept.sort_values("hire_rate")
ax.barh(dd.index, dd["hire_rate"], color=ACCENT)
ax.set_xlabel("Hire rate (%)"); ax.set_title("Hire rate by department")
plt.tight_layout(); plt.savefig("03_dept_hire_rate.png"); #plt.close()
plt.show()



In [ ]:
# Monthly applications trend
df["month"] = df["application_date"].dt.to_period("M").dt.to_timestamp()
monthly = df.dropna(subset=["month"]).groupby("month").agg(
    applications=("application_id","count"), hires=("hired","sum"))
fig, ax = plt.subplots(figsize=(9,4.5))
ax.plot(monthly.index, monthly["applications"], marker="o", color=ACCENT, label="Applications")
ax.plot(monthly.index, monthly["hires"], marker="s", color="#C0504D", label="Hires")
ax.set_title("Monthly applications and hires"); ax.legend(); ax.set_ylabel("Count")
plt.tight_layout(); plt.savefig("04_monthly_trend.png"); #plt.close()
plt.show()

In [ ]:
# Time to hire distribution
fig, ax = plt.subplots(figsize=(8,4.5))
ax.hist(tth, bins=20, color=ACCENT, alpha=.85)
ax.axvline(tth.median(), color="#C0504D", ls="--", label=f"Median {tth.median():.0f}d")
ax.set_title("Time-to-hire distribution"); ax.set_xlabel("Days"); ax.legend()
plt.tight_layout(); plt.savefig("05_tth_distribution.png"); #plt.close()
plt.show()